In [10]:
import numpy as np
import pandas as pd
import torch
from torch.testing import assert_close

# ==========================================
# 0. BLUEPRINTS FOR YOUR PROBLEM 1 METHODS
# ==========================================

def solve_theta_ode(q_grid, t_grid, gamma, sigma, T):
    """
    Placeholder for your Problem 1 backward integrator.
    Emits a theta grid of shape (len(t_grid), len(q_grid)).
    """
    Nt = len(t_grid)
    Nq = len(q_grid)
    return torch.ones((Nt, Nq), dtype=torch.float64)

def extract_quotes_from_theta(S, q_grid, t_grid, theta, gamma, kappa, sigma):
    """
    Placeholder to extract numerical r and delta from the theta grid.
    """
    tau = 1.0 - t_grid
    r_num = S - q_grid * gamma * (sigma ** 2) * tau
    log_term = torch.log(torch.tensor(1.0 + gamma / kappa))
    delta_num = (1.0 / gamma) * log_term + (gamma * (sigma ** 2) / 2.0) * tau
    return r_num, delta_num


# ==========================================
# 1. PROBLEM 2 CLOSED-FORM IMPLEMENTATION
# ==========================================

def get_asymptotic_quotes(
    S: torch.Tensor,
    q: torch.Tensor,
    gamma: float,
    sigma: float,
    T: float,
    t: torch.Tensor,
    kappa: float
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Computes the closed-form asymptotic reservation price (r)
    and half-spread (delta) for the Avellaneda-Stoikov model.
    """
    tau = T - t
    r = S - q * gamma * (sigma ** 2) * tau

    log_term = torch.log(torch.tensor(1.0 + gamma / kappa, dtype=tau.dtype, device=tau.device))
    delta = (1.0 / gamma) * log_term + (gamma * (sigma ** 2) / 2.0) * tau

    return r, delta


# ==========================================
# 2. REAL-WORLD DATA PROCESSING PIPELINE
# ==========================================

def load_market_parameters_from_parquet(file_path: str):
    """
    Loads a real parquet file, computes the mid-price series,
    and extracts the initial mid-price (S) and volatility (sigma).
    """
    print(f"Reading market data from {file_path}...")
    df = pd.read_parquet(file_path)

    # Adaptive Column Mapping
    df.columns = [col.lower() for col in df.columns]

    # Checking for specific column names
    if 'bid1_px' in df.columns and 'ask1_px' in df.columns:
        df['mid_price'] = (df['bid1_px'] + df['ask1_px']) / 2.0
    elif 'bid' in df.columns and 'ask' in df.columns:
        df['mid_price'] = (df['bid'] + df['ask']) / 2.0
    elif 'price' in df.columns:
        df['mid_price'] = df['price']
    else:
        print("Columns found:", df.columns.tolist())
        raise KeyError("Could not automatically locate price, mid_price, or bid/ask pairs.")

    # Drop any missing values
    mid_prices = df['mid_price'].dropna().values

    # --- Catch illiquid data before it crashes ---
    if len(mid_prices) < 2:
        print(f"[!] Warning: Only found {len(mid_prices)} valid data points after cleaning.")
        print("[!] Not enough data to calculate standard deviation. Defaulting to standard fallback.")

        # If there's 1 point, use it for S. Otherwise, guess 68,000 for this specific file.
        fallback_S = float(mid_prices[0]) if len(mid_prices) == 1 else 68000.0
        return fallback_S, 0.0004

    # Convert to log returns
    log_returns = np.diff(np.log(mid_prices))

    # Calculate step-wise volatility (sigma)
    sigma_estimated = np.std(log_returns) + 1e-8
    S_initial = mid_prices[0]

    print(f"Successfully processed {len(mid_prices)} ticks.")
    print(f"Calculated Initial Mid-Price (S): {S_initial:.2f}")
    print(f"Calculated Step-Volatility (sigma): {sigma_estimated:.8f}")

    return float(S_initial), float(sigma_estimated)


# ==========================================
# 3. MAIN RUNNER & TEST EXECUTION
# ==========================================

if __name__ == "__main__":
    # 1. Target the specific file uploaded to Colab
    sample_file = "BTCUSD-17JUN26-68000-P.parquet"

    try:
        S_val, sigma_val = load_market_parameters_from_parquet(sample_file)
    except FileNotFoundError:
        print(f"\n[!] File '{sample_file}' not found in local path.")
        S_val, sigma_val = 68000.0, 0.0004  # Fallbacks

    # 2. Setup structural model parameters
    gamma = 0.1     # Risk aversion
    kappa = 1.5     # Order book liquidity density
    T = 1.0         # Trading horizon terminal time

    # 3. Construct PyTorch Tensors and Grids
    S = torch.tensor(S_val, dtype=torch.float64)
    t_grid = torch.linspace(0.0, 0.9, steps=10, dtype=torch.float64)
    q_grid = torch.arange(-5.0, 6.0, dtype=torch.float64) # Inventory grid from -5 to +5

    # Meshgrid for vectorized calculations over space and time
    T_mesh, Q_mesh = torch.meshgrid(t_grid, q_grid, indexing='ij')

    # 4. Generate Closed-Form Solutions
    r_closed, delta_closed = get_asymptotic_quotes(
        S=S, q=Q_mesh, gamma=gamma, sigma=sigma_val, T=T, t=T_mesh, kappa=kappa
    )

    # 5. Generate Numerical Solutions via Problem 1 Functions
    # IMPORTANT: Update the mocked functions at the top of this script with your real math!
    theta_grid = solve_theta_ode(q_grid=q_grid, t_grid=t_grid, gamma=gamma, sigma=sigma_val, T=T)
    r_num, delta_num = extract_quotes_from_theta(
        S=S, q_grid=Q_mesh, t_grid=T_mesh, theta=theta_grid, gamma=gamma, kappa=kappa, sigma=sigma_val
    )

    # 6. Verify Agreement (The Pytest Assertion)
    print("\nValidating results...")
    try:
        assert_close(r_closed, r_num, rtol=1e-3, atol=1e-3)
        assert_close(delta_closed, delta_num, rtol=1e-3, atol=1e-3)
        print("✅ Success! The asymptotic closed-form quotes match the numerical solver.")
    except AssertionError as e:
        print("❌ Discrepancy found between asymptotic approximations and numerical solver.")
        print(e)

Reading market data from BTCUSD-17JUN26-68000-P.parquet...
[!] Warning: Only found 0 valid data points after cleaning.
[!] Not enough data to calculate standard deviation. Defaulting to standard fallback.

Validating results...
✅ Success! The asymptotic closed-form quotes match the numerical solver.


In [11]:

import numpy as np
import pandas as pd
import torch
from torch.testing import assert_close

# ==========================================
# 0. BLUEPRINTS FOR YOUR PROBLEM 1 METHODS
# ==========================================

def solve_theta_ode(q_grid, t_grid, gamma, sigma, T):
    """
    Placeholder for your Problem 1 backward integrator.
    Emits a theta grid of shape (len(t_grid), len(q_grid)).
    """
    Nt = len(t_grid)
    Nq = len(q_grid)
    return torch.ones((Nt, Nq), dtype=torch.float64)

def extract_quotes_from_theta(S, q_grid, t_grid, theta, gamma, kappa, sigma):
    """
    Placeholder to extract numerical r and delta from the theta grid.
    """
    tau = 1.0 - t_grid
    r_num = S - q_grid * gamma * (sigma ** 2) * tau
    log_term = torch.log(torch.tensor(1.0 + gamma / kappa))
    delta_num = (1.0 / gamma) * log_term + (gamma * (sigma ** 2) / 2.0) * tau
    return r_num, delta_num


# ==========================================
# 1. PROBLEM 2 CLOSED-FORM IMPLEMENTATION
# ==========================================

def get_asymptotic_quotes(
    S: torch.Tensor,
    q: torch.Tensor,
    gamma: float,
    sigma: float,
    T: float,
    t: torch.Tensor,
    kappa: float
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Computes the closed-form asymptotic reservation price (r)
    and half-spread (delta) for the Avellaneda-Stoikov model.
    """
    tau = T - t
    r = S - q * gamma * (sigma ** 2) * tau

    log_term = torch.log(torch.tensor(1.0 + gamma / kappa, dtype=tau.dtype, device=tau.device))
    delta = (1.0 / gamma) * log_term + (gamma * (sigma ** 2) / 2.0) * tau

    return r, delta


# ==========================================
# 2. REAL-WORLD DATA PROCESSING PIPELINE
# ==========================================

def load_market_parameters_from_parquet(file_path: str):
    """
    Loads a real parquet file, computes the mid-price series,
    and extracts the initial mid-price (S) and volatility (sigma).
    """
    print(f"Reading market data from {file_path}...")
    df = pd.read_parquet(file_path)

    # Adaptive Column Mapping
    df.columns = [col.lower() for col in df.columns]

    # Checking for specific column names
    if 'bid1_px' in df.columns and 'ask1_px' in df.columns:
        df['mid_price'] = (df['bid1_px'] + df['ask1_px']) / 2.0
    elif 'bid' in df.columns and 'ask' in df.columns:
        df['mid_price'] = (df['bid'] + df['ask']) / 2.0
    elif 'price' in df.columns:
        df['mid_price'] = df['price']
    else:
        print("Columns found:", df.columns.tolist())
        raise KeyError("Could not automatically locate price, mid_price, or bid/ask pairs.")

    # Drop any missing values
    mid_prices = df['mid_price'].dropna().values

    # --- Catch illiquid data before it crashes ---
    if len(mid_prices) < 2:
        print(f"[!] Warning: Only found {len(mid_prices)} valid data points after cleaning.")
        print("[!] Not enough data to calculate standard deviation. Defaulting to standard fallback.")

        # If there's 1 point, use it for S. Otherwise, guess 68,000 for this specific file.
        fallback_S = float(mid_prices[0]) if len(mid_prices) == 1 else 68000.0
        return fallback_S, 0.0004

    # Convert to log returns
    log_returns = np.diff(np.log(mid_prices))

    # Calculate step-wise volatility (sigma)
    sigma_estimated = np.std(log_returns) + 1e-8
    S_initial = mid_prices[0]

    print(f"Successfully processed {len(mid_prices)} ticks.")
    print(f"Calculated Initial Mid-Price (S): {S_initial:.2f}")
    print(f"Calculated Step-Volatility (sigma): {sigma_estimated:.8f}")

    return float(S_initial), float(sigma_estimated)


# ==========================================
# 3. MAIN RUNNER & TEST EXECUTION
# ==========================================

if __name__ == "__main__":
    # 1. Target the specific file uploaded to Colab
    sample_file = "BTC-USDT-SWAP_L2_2026-06-02_11h_12h.parquet"

    try:
        S_val, sigma_val = load_market_parameters_from_parquet(sample_file)
    except FileNotFoundError:
        print(f"\n[!] File '{sample_file}' not found in local path.")
        S_val, sigma_val = 68000.0, 0.0004  # Fallbacks

    # 2. Setup structural model parameters
    gamma = 0.1     # Risk aversion
    kappa = 1.5     # Order book liquidity density
    T = 1.0         # Trading horizon terminal time

    # 3. Construct PyTorch Tensors and Grids
    S = torch.tensor(S_val, dtype=torch.float64)
    t_grid = torch.linspace(0.0, 0.9, steps=10, dtype=torch.float64)
    q_grid = torch.arange(-5.0, 6.0, dtype=torch.float64) # Inventory grid from -5 to +5

    # Meshgrid for vectorized calculations over space and time
    T_mesh, Q_mesh = torch.meshgrid(t_grid, q_grid, indexing='ij')

    # 4. Generate Closed-Form Solutions
    r_closed, delta_closed = get_asymptotic_quotes(
        S=S, q=Q_mesh, gamma=gamma, sigma=sigma_val, T=T, t=T_mesh, kappa=kappa
    )

    # 5. Generate Numerical Solutions via Problem 1 Functions
    # IMPORTANT: Update the mocked functions at the top of this script with your real math!
    theta_grid = solve_theta_ode(q_grid=q_grid, t_grid=t_grid, gamma=gamma, sigma=sigma_val, T=T)
    r_num, delta_num = extract_quotes_from_theta(
        S=S, q_grid=Q_mesh, t_grid=T_mesh, theta=theta_grid, gamma=gamma, kappa=kappa, sigma=sigma_val
    )

    # 6. Verify Agreement (The Pytest Assertion)
    print("\nValidating results...")
    try:
        assert_close(r_closed, r_num, rtol=1e-3, atol=1e-3)
        assert_close(delta_closed, delta_num, rtol=1e-3, atol=1e-3)
        print("✅ Success! The asymptotic closed-form quotes match the numerical solver.")
    except AssertionError as e:
        print("❌ Discrepancy found between asymptotic approximations and numerical solver.")
        print(e)

Reading market data from BTC-USDT-SWAP_L2_2026-06-02_11h_12h.parquet...
Successfully processed 3433488 ticks.
Calculated Initial Mid-Price (S): 69625.00
Calculated Step-Volatility (sigma): 0.00029146

Validating results...
✅ Success! The asymptotic closed-form quotes match the numerical solver.


In [ ]:
import os
import glob
import numpy as np
import pandas as pd

def load_market_parameters_from_parquet(file_path: str):
    """
    Extracts the initial mid-price (S) and volatility (sigma) from a parquet file.
    """
    df = pd.read_parquet(file_path)
    df.columns = [col.lower() for col in df.columns]

    if 'bid1_px' in df.columns and 'ask1_px' in df.columns:
        df['mid_price'] = (df['bid1_px'] + df['ask1_px']) / 2.0
    elif 'bid' in df.columns and 'ask' in df.columns:
        df['mid_price'] = (df['bid'] + df['ask']) / 2.0
    elif 'price' in df.columns:
        df['mid_price'] = df['price']
    else:
        raise KeyError("Could not automatically locate price columns.")

    mid_prices = df['mid_price'].dropna().values

    if len(mid_prices) < 2:
        raise ValueError(f"Insufficient data: Only {len(mid_prices)} valid ticks found.")

    log_returns = np.diff(np.log(mid_prices))
    sigma_estimated = np.std(log_returns) + 1e-8
    S_initial = mid_prices[0]

    return float(S_initial), float(sigma_estimated), len(mid_prices)

def build_historical_parameter_surface(directory_path="."):
    """
    Iterates through all parquet files in a directory and compiles the market parameters.
    """
    print("Starting batch processing...")
    all_files = glob.glob(os.path.join(directory_path, "*.parquet"))

    if not all_files:
        print("No .parquet files found in the specified directory.")
        return pd.DataFrame()

    results = []

    for file in sorted(all_files):
        file_name = os.path.basename(file)
        try:
            S, sigma, tick_count = load_market_parameters_from_parquet(file)
            results.append({
                "File": file_name,
                "Ticks": tick_count,
                "Mid_Price_S": S,
                "Volatility_Sigma": sigma,
                "Status": "Success"
            })
            print(f"Processed: {file_name} | S: {S:.2f} | Sigma: {sigma:.6f}")

        except Exception as e:
            print(f"Skipped: {file_name} | Reason: {str(e)}")
            results.append({
                "File": file_name,
                "Ticks": 0,
                "Mid_Price_S": None,
                "Volatility_Sigma": None,
                "Status": f"Failed: {str(e)}"
            })

    # Convert to a clean master table
    master_df = pd.DataFrame(results)
    print("\nBatch processing complete!")
    return master_df

if __name__ == "__main__":
    # Run the loop on the current Colab directory
    master_parameter_table = build_historical_parameter_surface(".")

    # Display the final aggregated table
    display(master_parameter_table)